# Daily NTL Data -- Trinidad and Tobago

Last update: 2026-10-19. Ported from the Bolivia pipeline
(`2026_NWCST_BOL_Workshop/code/01_ntl.ipynb`); the previous version is in
the private project repository.

**Run with the `geo` kernel.** Settings live in `config.py` (same folder); the
functions live in `ntl_helpers.py`. Edit `config.py`, not this notebook.

What is different from the old notebook:

* **Targeted download.** `missing_targets` asks LAADS which days exist inside
  `cfg.NTL_DATE_RANGE`, diffs that against the tile names in `hfiles/h11v07/`, and
  fetches only the difference -- in parallel, with retries, behind a
  `cfg.NTL_MAX_FILES` cap that refuses an oversized run before a byte moves.
* **No re-reading.** A date already in `raw/ntl_shape.csv` / `raw/ntl_data.csv` is
  skipped on the date in the *filename*, so its `.h5` is never opened. That matters
  because `hfiles/` is Dropbox online-only: opening an evicted tile downloads it again.
* Outputs keep their existing names and columns, so `2_data_build` and `meta.csv`
  need no change.

In [ ]:
# ===========================================================================
# CONFIG LOADER  --  edit config.py, NOT this cell.
# Gives you `cfg` (every country knob) and `P` (the standard paths).
# ===========================================================================
import os, sys, importlib, importlib.util
import numpy as np
import pandas as pd
from tqdm import tqdm

CODE_DIR = os.getcwd()
_spec = importlib.util.spec_from_file_location("config", os.path.join(CODE_DIR, "config.py"))
cfg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(cfg)
P = cfg.derive_paths(CODE_DIR)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
import ntl_helpers
importlib.reload(ntl_helpers)      # pick up edits without restarting the kernel
from ntl_helpers import *

if not os.path.isdir(P["nl_root"]):
    raise FileNotFoundError(f"Tile store not found: {P['nl_root']} -- check cfg.NL_ROOT")

# Without GDAL's HDF5 driver nothing raises -- every tile just reads as empty
# and the run ends "fine" with no data. Checked before anything is downloaded.
check_hdf5_driver()

print(f"[CONFIG] {cfg.COUNTRY} ({cfg.CODE}/{cfg.GADM_ISO3})")
print(f"         tiles      : {cfg.TILES}   (download: {cfg.download_tiles()})")
print(f"         window     : {cfg.NTL_DATE_RANGE or 'ALL available dates'}")
print(f"         outputs    : raw/{cfg.NTL_SHAPE_CSV}, raw/{cfg.NTL_OUTPUT_CSV}")
print(f"         tile store : {P['nl_root']}")
print(f"         HDF5 driver: OK")

## Download

The NASA token is `cfg.NASA_TOKEN` in `config.py`. It expires after ~60 days; when
downloads start returning `401`, generate a new one at
<https://ladsweb.modaps.eosdis.nasa.gov/> (My Account -> Generate Token). With an
empty token the download is skipped and the rest runs on the tiles already on disk.

In [ ]:
api_key    = cfg.NASA_TOKEN
HAVE_TOKEN = bool(api_key) and not api_key.startswith("PASTE")

# Names only -- listing the folder does not pull online-only files back down.
inventory = {t: len(list_tile_rasters(P["nl_root"], [t], make=False)) for t in cfg.TILES}
print("Tiles on disk:")
for t, n in inventory.items():
    print(f"   {t}: {n:,} file(s)")

if not HAVE_TOKEN:
    print("\nNASA_TOKEN is empty in config.py -> skipping download.")

In [ ]:
from collections import Counter

# ntl_window() applies NTL_FIRST_DATE as a floor even when the range is None.
start_date, end_date = cfg.ntl_window()

_tiles = len(cfg.download_tiles())
if end_date is not None:
    _days  = (end_date - start_date).days + 1
    print(f"Window: {start_date.date()} -> {end_date.date()}  ({_days} days)")
    print(f"At most {_days * _tiles:,} files, about {_days * _tiles * 25 / 1024:.1f} GB "
          f"if none are on disk yet.")
else:
    print(f"Window: {start_date.date()} -> latest available (open end).")
    print(f"Capped at cfg.NTL_MAX_FILES = {cfg.NTL_MAX_FILES} files.")

# The download log lives with the tiles it describes.
log_path = os.path.join(P["nl_root"], "ntl_download_log.txt")

if HAVE_TOKEN:
    target = missing_targets(cfg.download_tiles(), api_key, P["nl_root"],
                             start_date=start_date, end_date=end_date)
    print()
    print(f"{len(target)} file(s) available on the server but missing on disk "
          f"(~{len(target) * 25 / 1024:.1f} GB to fetch).")
    if target:
        for _t, _n in sorted(Counter(t for t, _, _ in target).items()):
            print(f"   {_t}: {_n:,}")
        try:
            result = download_many(target, api_key, P["nl_root"],
                                   log_path=log_path, max_workers=8,
                                   max_files=cfg.NTL_MAX_FILES)
        except ntl_helpers.TooManyTargets as e:
            # Deliberate stop, not a crash: nothing has been downloaded.
            print()
            print("STOPPED --", e)
            result = {}
        missing = result.get("unavailable", 0) + result.get("failed", 0)
        if result:
            print("All files downloaded." if missing == 0
                  else f"{missing} file(s) not downloaded -- see {log_path}")
    else:
        print("Nothing missing: up to date.")
else:
    print("No token -> skipped. Working with what is already on disk.")

## Country boundary

GADM level 0 (the national outline), cached in `code/shapes/` and downloaded
only the first time.

In [ ]:
import geopandas as gpd
import urllib.request

shape_dir = os.path.join(CODE_DIR, "shapes")
os.makedirs(shape_dir, exist_ok=True)
shape_zip = os.path.join(shape_dir, cfg.GADM_ZIP)
shape_url = "https://geodata.ucdavis.edu/gadm/gadm4.1/shp/" + cfg.GADM_ZIP

if not os.path.exists(shape_zip):
    print("Downloading boundary (once):", shape_url)
    urllib.request.urlretrieve(shape_url, shape_zip)
print("Boundary:", shape_zip)

gdf         = gpd.read_file(f"zip://{shape_zip}!gadm41_{cfg.GADM_ISO3}_0.shp")
gdf_country = gdf[gdf.geometry.notnull()].dissolve().to_crs("EPSG:4326")
print(f"bounds: {tuple(round(b, 2) for b in gdf_country.total_bounds)}")

gdf_country.explore(tiles="CartoDB dark_matter", zoom=9)

## Country series -> `raw/ntl_shape.csv`

Mean and total radiance over the national polygon, one row per night. Only dates
not yet in the CSV are read.

In [ ]:
_ws, _we = cfg.ntl_window()

shape_csv = os.path.join(P["raw"], cfg.NTL_SHAPE_CSV)
files = rasters_in_range(P["nl_root"], list(cfg.TILES), start=_ws, end=_we)

# RESUME: filtered on the date in the filename, so skipped files are never opened.
done    = shape_dates_done(shape_csv, len(cfg.TILES), out_cols=cfg.NTL_SHAPE_COLS)
pending = pending_rasters(files, done, label="Country NTL")
if cfg.NTL_MAX_FILES and len(pending) > cfg.NTL_MAX_FILES:
    raise RuntimeError(f"{len(pending):,} tiles to read (cfg.NTL_MAX_FILES = {cfg.NTL_MAX_FILES}) "
                       f"-- is raw/{cfg.NTL_SHAPE_CSV} missing? Re-reading the archive "
                       "rehydrates every online-only tile.")

recs = []
for _path, _dt in tqdm(pending, desc="Country NTL"):
    try:
        recs.append(processHD5_shape(_path, 2, P["ntl_tmp"], gdf_country, _dt))
    except Exception as e:
        print(f"  skip {os.path.basename(_path)}: {type(e).__name__}: {e}")

# Merges the new dates into the CSV (keeping its shape_mean/shape_sum columns)
# and returns the WHOLE series under the internal ntl_* names.
shape_df = update_shape_csv(recs, shape_csv, len(cfg.TILES), out_cols=cfg.NTL_SHAPE_COLS)

if not len(shape_df):
    raise RuntimeError("No data: hfiles/<tile>/ has nothing in this window.")

print()
print(f"{len(shape_df):,} row(s) in raw/{cfg.NTL_SHAPE_CSV}  ({len(recs)} file(s) read this run), "
      f"{shape_df.index.min().date()} to {shape_df.index.max().date()}")
shape_df[["ntl_mean", "ntl_sum"]].tail()

In [ ]:
import matplotlib.pyplot as plt

marks = ([(pd.Timestamp(cfg.NTL_EVENT_DATE), cfg.NTL_EVENT_LABEL or "event")]
         if cfg.NTL_EVENT_DATE else [])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(shape_df.index, shape_df["ntl_mean"], color="#005BAC", linewidth=0.8,
        alpha=0.5, label="daily")
ax.plot(shape_df["ntl_mean"].rolling(30, min_periods=10).mean(), color="#005BAC",
        linewidth=1.8, label="30-day mean")
for _d, _lab in marks:
    ax.axvline(_d, color="#CC0000", linewidth=1.5, linestyle="--", zorder=3)
    ax.text(_d, ax.get_ylim()[1], f" {_lab}", color="#CC0000", fontsize=9,
            va="top", ha="left")

ax.set_title(f"{cfg.COUNTRY} - nighttime lights, country mean", fontsize=13, weight="bold")
ax.set_ylabel("nW/cm2/sr")
ax.grid(axis="y", linestyle=":", alpha=0.5)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
print("saved ->", save_fig(fig, P, "ntl_country_mean", notebook="1_data_ntl"))
plt.show()

## Before and after, on a map

With `cfg.NTL_EVENT_DATE` set: event +/- `NTL_EVENT_WINDOW` days. Without one:
the first and last extracted date of the window. This is the one step that opens
a tile after extraction -- two files, ~50 MB if they were evicted.

In [ ]:
import contextily as cx
import colorcet as cc

if cfg.NTL_EVENT_DATE:
    _ev = pd.Timestamp(cfg.NTL_EVENT_DATE)
    d_before = _ev - pd.Timedelta(days=cfg.NTL_EVENT_WINDOW)
    d_after  = _ev + pd.Timedelta(days=cfg.NTL_EVENT_WINDOW)
    _lab = cfg.NTL_EVENT_LABEL or "event"
    lab_before = f"{cfg.NTL_EVENT_WINDOW} days before {_lab}"
    lab_after  = f"{cfg.NTL_EVENT_WINDOW} days after {_lab}"
else:
    # The window, not the whole history: the first date would otherwise be 2012.
    _in = shape_df.index[shape_df.index >= pd.Timestamp(_ws)]
    if _we is not None:
        _in = _in[_in <= pd.Timestamp(_we)]
    d_before, d_after = (_in.min(), _in.max()) if len(_in) else (shape_df.index[-1],) * 2
    lab_before, lab_after = "window start", "window end"

BASEMAP   = True        # False = plain background, fully offline
DRAW_MAPS = getattr(cfg, "NTL_DRAW_MAPS", True)

f_before = find_h5s_for_date(d_before, P["nl_root"], list(cfg.TILES)) if DRAW_MAPS else []
f_after  = find_h5s_for_date(d_after,  P["nl_root"], list(cfg.TILES)) if DRAW_MAPS else []
if DRAW_MAPS:
    print(f"before ({d_before.date()}): {len(f_before)}/{len(cfg.TILES)} tile(s)")
    print(f"after  ({d_after.date()}): {len(f_after)}/{len(cfg.TILES)} tile(s)")

_maps_ok = DRAW_MAPS and bool(f_before) and bool(f_after)
if _maps_ok:
    try:
        arr_b, lons, lats = raster_to_array(f_before, 2, gdf_country, P["ntl_tmp"])
        arr_a, _, _       = raster_to_array(f_after,  2, gdf_country, P["ntl_tmp"])
    except Exception as e:
        _maps_ok = False
        print(f"Could not read the tiles for those dates ({type(e).__name__}: {e}). "
              "The series above is unaffected -- it came from the CSV.")

if not _maps_ok:
    print("cfg.NTL_DRAW_MAPS is False -> maps skipped, no tile opened."
          if not DRAW_MAPS else "Skipping the before/after maps.")
else:
    both = np.concatenate([arr_b[~np.isnan(arr_b)], arr_a[~np.isnan(arr_a)]])
    vmin, vmax = np.nanpercentile(both, 2), np.nanpercentile(both, 98)

    # imshow on the regular grid (pcolormesh draws one polygon per pixel).
    dx, dy = abs(lons[1] - lons[0]), abs(lats[1] - lats[0])
    extent = [lons[0] - dx / 2, lons[-1] + dx / 2, lats[-1] - dy / 2, lats[0] + dy / 2]

    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    fig.subplots_adjust(bottom=0.15)
    titles = [f"{lab_before}\n({d_before.strftime('%b %d, %Y')})",
              f"{lab_after}\n({d_after.strftime('%b %d, %Y')})"]
    for ax, arr, title in zip(axes, [arr_b, arr_a], titles):
        # Explicit zorder: add_basemap() draws later and would hide the lights.
        im = ax.imshow(arr, cmap=cc.cm.bmy, vmin=vmin, vmax=vmax,
                       extent=extent, origin="upper", interpolation="nearest", zorder=2)
        gdf_country.boundary.plot(ax=ax, color="#444444", linewidth=0.8, zorder=3)
        ax.set_title(title, fontsize=13, weight="bold")
        if BASEMAP:
            try:
                # Esri: CartoDB now stamps unkeyed tiles "API KEY REQUIRED" and
                # OSM answers contextily's user agent with a 403 tile.
                cx.add_basemap(ax, source=cx.providers.Esri.WorldGrayCanvas,
                               crs="EPSG:4326", attribution_size=6, zorder=0)
            except Exception as e:
                print(f"  basemap unavailable ({type(e).__name__}) -- plotting without it")
        ax.set_xlim(extent[0], extent[1]); ax.set_ylim(extent[2], extent[3])
        ax.set_aspect("equal")
        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")

    cbar_ax = fig.add_axes([0.25, 0.05, 0.5, 0.025])
    fig.colorbar(im, cax=cbar_ax, orientation="horizontal", label="NTL radiance (nW/cm2/sr)")
    fig.suptitle(f"Nighttime light radiance - {cfg.COUNTRY}", fontsize=15, weight="bold")
    fig.text(0.01, 0.01, "Source: NASA Black Marble VNP46A2", fontsize=10)
    print("saved ->", save_fig(fig, P, "ntl_map_before_after", notebook="1_data_ntl"))
    plt.show()

## Sites -> `raw/ntl_data.csv`

`raw/Coordinates.xlsx` lists the sites (sector, location, city, lat, lon). Two
readings per site per night: `DNBvalue1` (the pixel) and `DNBvalue3` (3x3 mean,
the one to prefer). A date counts as done only when every site has a row, so a
site added to `Coordinates.xlsx` is backfilled across the whole history -- which
means re-reading every tile since 2012. Add sites knowingly.

In [ ]:
coord_path = os.path.join(P["raw"], cfg.COORDINATES_XLSX)
site_csv   = os.path.join(P["raw"], cfg.NTL_OUTPUT_CSV)

locations = pd.read_excel(coord_path)
# Strip accents and stray whitespace so names stay stable merge keys.
for col in ("location", "city", "sector"):
    locations[col] = (locations[col].fillna("").astype(str)
                        .str.normalize("NFKD")
                        .str.encode("ascii", errors="ignore").str.decode("utf-8")
                        .str.strip())

coords = locations[["sector", "location", "city", "lat", "lon"]].to_dict("records")
print(f"{len(coords)} site(s) in {cfg.COORDINATES_XLSX}; sectors: "
      + ", ".join(sorted(locations["sector"].unique())))

done    = site_dates_done(site_csv, locations["location"])
pending = pending_rasters(files, done, label="Site NTL")

# Guard: a renamed or new site makes EVERY date pending. Refuse to rehydrate the
# whole online-only archive by accident.
if cfg.NTL_MAX_FILES and len(pending) > cfg.NTL_MAX_FILES:
    raise RuntimeError(f"{len(pending):,} tiles to read for the sites (cfg.NTL_MAX_FILES = "
                       f"{cfg.NTL_MAX_FILES}) -- did a location name change in "
                       "Coordinates.xlsx? Check before re-reading the archive.")

rows = []
for _path, _dt in tqdm(pending, desc="Site NTL"):
    try:
        rows.extend(processHD5(_path, 2, P["ntl_tmp"], coords, _dt))
    except Exception as e:
        print(f"  skip {os.path.basename(_path)}: {type(e).__name__}: {e}")

site_df = update_site_csv(rows, site_csv)
print(f"{len(site_df):,} site-day(s) in raw/{cfg.NTL_OUTPUT_CSV} ({len(rows):,} written this run), "
      f"last date {site_df['date'].max().date()}")
site_df.tail()

In [ ]:
# Mean brightness by sector, last 12 months.
_recent = site_df[site_df["date"] >= site_df["date"].max() - pd.Timedelta(days=365)]
by_sector = _recent.groupby(["date", "sector"])["DNBvalue3"].mean().unstack("sector")

fig, ax = plt.subplots(figsize=(12, 4))
by_sector.rolling(7, min_periods=1).mean().plot(ax=ax, linewidth=1.4)
ax.set_title(f"{cfg.COUNTRY} - site radiance by sector (7-day mean)", fontsize=13, weight="bold")
ax.set_ylabel("nW/cm2/sr"); ax.set_xlabel("")
ax.grid(axis="y", linestyle=":", alpha=0.5)
ax.legend(frameon=False, ncol=3, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
print("saved ->", save_fig(fig, P, "ntl_by_sector", notebook="1_data_ntl"))
plt.show()